# 04 日 K + 周 K 一起训练

每个交易日两张图：当天的 20 日 K，加上最近一根已经走完的周 K。两个卷积塔不共享权重，特征拼接后一个线性输出。不要把两个模型的分数做平均。

目标仍是未来 5 日超额。截尾、标准化、10 个 epoch、按验证集 Spearman 留模型，和日 K 那一版一样。

日 K 读 `k_pictures_gray_body`。周 K 读 `k_pictures_gray_fuse_week`。权重和预测写到 `k_pictures_gray_fuse`。

先 `N_SEEDS = 1`。对照实验把 `config.yaml` 的 `fuse.inputs` 改成 `[day]`，并把 `output_remote` 改成另一个目录，再训一遍。那样只有周 K 这一路不同。

In [ ]:
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
candidates = [cwd, cwd / "change_rgb_i20"]
candidates.extend(cwd.parents)
for parent in [cwd, *cwd.parents]:
    candidates.append(parent / "change_rgb_i20")
seen = set()
root = None
for p in candidates:
    key = str(p)
    if key in seen:
        continue
    seen.add(key)
    if (p / "helpers.py").exists() and (p / "config.yaml").exists() and (p / "train_lib.py").exists():
        text = (p / "config.yaml").read_text(encoding="utf-8")
        if "task: regression" in text:
            root = p
            break
if root is None:
    raise FileNotFoundError(
        f"找不到 change_rgb_i20（config.yaml 里要有 task: regression）。cwd={cwd}"
    )
missing = [name for name in ("cnn.py", "train_lib.py", "backtest_lib.py") if not (root / name).exists()]
if missing:
    raise FileNotFoundError(
        f"{root} 缺少 {missing}。这些文件和 helpers.py 放在同一目录，不要再套一层。"
    )
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import torch
from helpers import notebook_bootstrap
from train_lib import predict_window, train_window

ROOT, cfg, paths = notebook_bootstrap()
print("ROOT   =", ROOT)
print("task   =", cfg.cnn.task, "epochs =", cfg.cnn.max_epochs, "lr =", cfg.cnn.optimizer.lr)
print("day    =", cfg.paths.images_remote)
print("week   =", cfg.paths.week_images_remote)
print("output =", cfg.paths.output_remote)
print("inputs =", list(cfg.fuse.inputs))
print("cuda   =", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

## 训练

`SKIP_EXISTING=True` 时已有最终的 `seed_k.pt` 会跳过。每个 epoch 会写 `seed_k.running.pt`，里面是到目前为止验证集 Spearman 最高的权重。10 轮全部跑完后，`seed_k.pt` 才是最终留下的那一轮，`running` 文件会删掉。

`WINDOW_KEY` 用 `"I20"` 或 `"I60"`，须与 03 生成的图一致。

In [ ]:
WINDOW_KEY = "I20"   # 或 "I60"，须与已生成的图一致
N_SEEDS = 1
SKIP_EXISTING = True

results = train_window(
    cfg, paths, WINDOW_KEY,
    n_seeds=N_SEEDS,
    num_workers=0,
    skip_existing=SKIP_EXISTING,
    pin_memory=False,
)
print(results)

## 样本外预测（test split，已有种子平均）

`pred` 是预测的 5 日超额。多个种子先各自还原成超额，再取平均。试通后把上面的 `N_SEEDS` 改成 5，再跑训练和这一格。然后打开 `05_plot_top10.ipynb`。

In [ ]:
pred = predict_window(cfg, paths, WINDOW_KEY, num_workers=0)
display(pred.head())